In [1]:
import json
import base64 
from ChunkCaptioner import ImageCaptioner
from pathlib import Path


/home/prime/miniconda3/envs/preprocess/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_image_size(image_b64):
    decoded_image = base64.b64decode(image_b64)
    size_kb = len(decoded_image) / 1024
    return size_kb

In [3]:
def make_Pil_image(image_b64):
    from PIL import Image
    from io import BytesIO
    decoded_image = base64.b64decode(image_b64)
    image = Image.open(BytesIO(decoded_image))
    return image
def show_image(image_b64):
    from IPython.display import display, Image
    display(make_Pil_image(image_b64))

In [4]:
sys_prompt = """
ROLE:
You are a technical documentation assistant generating captions for ORAN and 5G network diagrams.

TASK:
Generate a concise caption describing the image for technical documentation.

MANDATORY RULES:

1. ENTITY IDENTIFICATION
- Multiple boxes with identical labels = ONE entity at different time points
- Never say "first X" or "second X" - only "X"

2. ARROW DIRECTION
- CRITICAL: Trace arrows from tail to arrowhead
- "←" means flow FROM RIGHT TO LEFT
- "→" means flow FROM LEFT TO RIGHT
- Describe as "from X to Y"
- Two opposite arrows = two distinct messages (not bidirectional)
- Only call bidirectional if single arrow has arrowheads on both ends

3. MESSAGE SEQUENCING
- List ALL messages in top-to-bottom order
- Format: "Message Name" sent from [Source] to [Destination]
- Skip duplicate message interactions

4. CONTENT RESTRICTIONS
- Use ONLY visible information
- Use acronyms exactly as shown (don't expand)
- No assumptions or external knowledge
- Professional language only

EXAMPLE 1:
Input: Near-RT RIC and E2 Node sequence diagram
Output: The diagram shows interactions between a Near-RT RIC and an E2 Node:
1. "RIC SUBSCRIPTION REQUEST" from Near-RT RIC to E2 Node
2. "RIC SUBSCRIPTION FAILURE" from E2 Node to Near-RT RIC

EXAMPLE 2:
Input: Near-RT RIC and E2 Node single message
Output: The diagram shows: "RIC SUBSCRIPTION REQUEST" from E2 Node to Near-RT RIC

EXAMPLE 3:
Input: Heading/title only
Output: ""
 """

json_folder = "test"  # Change this to your folder path
json_files = list(Path(json_folder).glob("*.json"))
print(f"Found {len(json_files)} JSON files")

blocktypes = ["Figure", "FigureGroup"] # if some are missing include "Pictures" as a blocktype

captioner = ImageCaptioner(system_prompt=sys_prompt)


for json_path in json_files:
    print(f"\nProcessing: {json_path.name}")
    try:
        with open(json_path, 'r') as file:
            data = json.load(file)
        for block in data:
            if block["block_type"] in blocktypes:
                description = captioner.generate_caption(base64_string=block["images"])
                block['description'] = description
                #show_image(img_b64)  # Display the image in the notebook
                print(description)  # Output the generated caption
    
        with open(json_path, 'w') as file:
            json.dump(data, file, indent=4)
        print(f"  ✓ Updated {json_path.name}")
    except Exception as e:
        print(f"  ✗ Failed to process {json_path.name}: {e}")
        continue


Found 2 JSON files
Loading model from BytedanceDouyinContent/SAIL-VL-1d6-8B...


You are using a model of type internvl_chat to instantiate a model of type sailvl. This is not supported for all configurations of models and can yield errors.
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00,  5.99it/s]


Model loaded successfully.

Processing: O-RAN-WG1-CCIN-TR-R004-v01.00_cleaned.json


Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


The diagram depicts cloud resources integration in an RAN deployment and applications sharingability alongside other computing applications:

1. "Cloud Resources Integration" from apps within the cloud and RAN to support sharingability
2
The diagram shows an XR signal routing through ORAN and 5G network nodes:
1. "XR SIGNAL ROUTING" from Blue Node 1 to Red XR Signal
22. "XR SIGNAL ROUTING" from Red XR Signal to Green Node 4
3. "XR SIGNAL ROUTING" from Red XR Signal to Yellow Node 3
4. "XR SIGNAL ROUTING" from Green Node 4 to Red XR Signal
5. "XR SIGNAL ROUTING" from Yellow Node 3 to Red XR Signal
6. "XR SIGNAL ROUTING" from Pink Node 2 to Red XR Signal
7. "XR SIGNAL ROUTING" from Blue Node 1 to Red XR Signal
8. "XR SIGNAL ROUTING" from Pink Node 2 to Red XR Signal
The diagram illustrates the interactions within an ORAN network context:
1. "2O2 Data Collection Request/Subscribe" from Service Management and Orchestration (SMO) to IMS/DMS
5. "" from IMS/DMS to Service Management and Orche